# YOLO Ultralytics — Real-Time Object Detection

## What Is This Notebook About?

**Classification** says: "There is a cat in this image."  
**Object Detection** says: "There is a cat at position (x=120, y=80, width=150, height=200), a dog at (x=300, y=50, ...), and a car at (x=10, y=200, ...)."  

**YOLO (You Only Look Once)** is the world's most popular object detection framework — fast enough to run at 60+ FPS on a laptop, accurate enough for production systems.

Ultralytics YOLO (v5/v8/v11) wraps everything into a remarkably simple API: 3 lines of code to detect objects in any image.

---

## Real-World Applications

| Industry | YOLO Application |
|---|---|
| Autonomous vehicles | Detect cars, pedestrians, cyclists, signs in real time |
| Retail | Automatic checkout (detect products on conveyor belt) |
| Security | Intrusion detection, crowd counting |
| Sports | Player tracking, ball trajectory, offside detection |
| Healthcare | Detect polyps in endoscopy, cell counting |
| Agriculture | Detect diseased plants, count fruit on trees |
| Manufacturing | Defect detection on assembly lines |

---

## Prerequisites

- Basic Python (lists, loops)
- OpenCV notebook (what images are, drawing)
- Torchvision notebook (what CNNs do)

---

## Table of Contents

1. [Setup & YOLO Model Sizes](#1-setup)
2. [How YOLO Works (Plain English)](#2-how-yolo-works)
3. [Running Inference — Detect in Any Image](#3-inference)
4. [Understanding YOLO Output — Boxes, Scores, Classes](#4-output)
5. [YOLO Tasks — Detection, Segmentation, Classification, Pose](#5-tasks)
6. [Training YOLO on Custom Data](#6-training)
7. [Model Export & Deployment](#7-export)
8. [Mini Project — Vehicle Counter for Traffic Analysis](#8-mini-project)
9. [Common Pitfalls](#9-pitfalls)
10. [Interview Q&A](#10-interview)
11. [Resources](#11-resources)
12. [Summary](#12-summary)

---
## 1. Setup & YOLO Model Sizes <a id='1-setup'></a>

In [ ]:
# pip install ultralytics opencv-python matplotlib numpy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.cm as cm
import cv2
import time
import warnings
warnings.filterwarnings('ignore')

try:
    from ultralytics import YOLO
    import torch
    YOLO_AVAILABLE = True
    print(f"Ultralytics YOLO available")
    device = 'cuda' if torch.cuda.is_available() else \
             'mps'  if torch.backends.mps.is_available() else 'cpu'
    print(f"Device: {device}")
except ImportError:
    YOLO_AVAILABLE = False
    print("ultralytics not installed — pip install ultralytics")
    print("All examples show code structure; simulated output shown where needed.")
    device = 'cpu'

np.random.seed(42)
print("\nReady!")

In [ ]:
# ── YOLO model size comparison ────────────────────────────────────────────────

yolo_models = [
    ('YOLOv8n (nano)',  'yolov8n.pt',  3.2,   8.7,  80,  'Edge devices, mobile'),
    ('YOLOv8s (small)', 'yolov8s.pt', 11.2,  28.6,  50,  'Fast CPU inference'),
    ('YOLOv8m (medium)','yolov8m.pt', 25.9,  78.9,  35,  'Balanced'),
    ('YOLOv8l (large)', 'yolov8l.pt', 43.7, 165.2,  20,  'High accuracy, GPU'),
    ('YOLOv8x (xlarge)','yolov8x.pt', 68.2, 257.8,  12,  'Maximum accuracy'),
    ('YOLOv11n',        'yolo11n.pt',  2.6,   6.5,  100, 'Newest, fastest'),
    ('YOLOv11x',        'yolo11x.pt', 56.9, 194.9,   14, 'Newest, best accuracy'),
]

print(f"{'Model':22s} {'Params(M)':10s} {'GFLOPs':8s} {'FPS(CPU)':9s} {'Best For'}")
print("─" * 75)
for name, weight, params, gflops, fps, notes in yolo_models:
    print(f"{name:22s} {params:10.1f} {gflops:8.1f} {fps:>9}  {notes}")

print("\nChoosing the right size:")
print("  Raspberry Pi / phone: yolov8n or yolo11n")
print("  Laptop (CPU):         yolov8s or yolov8m")
print("  Server/cloud GPU:     yolov8l or yolov8x")
print("  Default for learning: yolov8n (fastest download, still capable)")

---
## 2. How YOLO Works (Plain English) <a id='2-how-yolo-works'></a>

### The Newspaper Reporter Analogy

A reporter looks at ONE photo and instantly reports: "A dog in the top-left, a cat in the center, and a bicycle in the bottom-right." They don't scan the image region-by-region (that's the old Sliding Window approach) — they see the whole image **at once** and tell you everything in one pass.

That's YOLO: **You Only Look Once** — one forward pass → all detections.

### Architecture (YOLOv8)

```
Input Image (640×640)
    ↓
Backbone (CSPDarknet) — extract features at multiple scales
    ↓
Neck (FPN + PAN) — merge features from different scales
    ↓
Head — for each of 8400 anchor-free grid cells, predict:
    • Bounding box: (x_center, y_center, width, height)
    • Objectness + class probabilities: P(class | object)
    ↓
NMS (Non-Maximum Suppression) — remove duplicate boxes
    ↓
Final Detections: [(x1,y1,x2,y2), confidence, class_id]
```

### NMS (Non-Maximum Suppression)

Multiple grid cells often detect the same object. NMS keeps only the best box:
1. Sort boxes by confidence score (highest first)
2. Keep the highest-confidence box
3. Remove all boxes that overlap >50% with the kept box (IoU threshold)
4. Repeat with remaining boxes

**IoU (Intersection over Union)** = overlap between two boxes:  
`IoU = Area(intersection) / Area(union)`  
IoU = 0: no overlap. IoU = 1: perfect match.

### YOLO vs Two-Stage Detectors (Faster R-CNN)

| Aspect | YOLO (one-stage) | Faster R-CNN (two-stage) |
|---|---|---|
| Speed | 30-100 FPS | 5-15 FPS |
| Accuracy | Very good | Slightly better |
| Small objects | OK | Better |
| Use case | Real-time, edge | High-accuracy, server |

In [ ]:
# ── Visualize IoU concept ─────────────────────────────────────────────────────

def compute_iou(box1, box2):
    """Compute IoU between two boxes [x1,y1,x2,y2]."""
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection = max(0, x2-x1) * max(0, y2-y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - intersection
    return intersection / union if union > 0 else 0


# Demo: different IoU scenarios
scenarios = [
    ('No overlap (IoU=0)',        [10,10,50,50], [60,60,100,100]),
    ('Partial overlap',           [10,10,60,60], [40,40,90,90]),
    ('High overlap',              [10,10,80,80], [15,15,85,85]),
    ('Perfect match (IoU=1)',     [10,10,80,80], [10,10,80,80]),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colors = ['#3498db', '#e74c3c']

for ax, (title, b1, b2) in zip(axes, scenarios):
    ax.set_xlim(0, 110); ax.set_ylim(0, 110)
    ax.set_aspect('equal')

    for box, color, label in [(b1, colors[0], 'Predicted'), (b2, colors[1], 'Ground Truth')]:
        rect = patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1],
                                   linewidth=2, edgecolor=color, facecolor=color, alpha=0.25, label=label)
        ax.add_patch(rect)

    iou = compute_iou(b1, b2)
    ax.set_title(f'{title}\nIoU = {iou:.2f}', fontsize=9)
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

plt.suptitle('IoU (Intersection over Union) — key metric for detection quality', fontsize=12)
plt.tight_layout()
plt.show()

print("IoU thresholds commonly used:")
print("  IoU ≥ 0.5: 'correct detection' (COCO AP50 metric)")
print("  IoU ≥ 0.75: 'strict correct detection' (COCO AP75 metric)")
print("  NMS threshold: typically 0.45–0.65 (suppress overlapping boxes)")

---
## 3. Running Inference — Detect in Any Image <a id='3-inference'></a>

In [ ]:
# ── Load YOLO model ───────────────────────────────────────────────────────────

if YOLO_AVAILABLE:
    print("Loading YOLOv8n (nano — smallest, fastest)...")
    print("(First run downloads weights ~6MB from GitHub)")
    model = YOLO('yolov8n.pt')   # downloads automatically
    print(f"Model loaded! Classes: {len(model.names)} COCO categories")
    print(f"First 10 classes: {[model.names[i] for i in range(10)]}")
else:
    print("[SIMULATION] YOLO model structure:")
    COCO_CLASSES = [
        'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train',
        'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign',
        'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep',
        'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella',
        'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard',
        'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard',
        'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork',
        'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
        'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair',
        'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv',
        'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave',
        'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase',
        'scissors', 'teddy bear', 'hair drier', 'toothbrush'
    ]
    print(f"COCO dataset: {len(COCO_CLASSES)} classes")
    print(f"First 10: {COCO_CLASSES[:10]}")

In [ ]:
# ── Create synthetic scene for detection demo ─────────────────────────────────

def create_scene(width=640, height=480):
    """Create a synthetic scene with labeled objects for demo."""
    scene = np.ones((height, width, 3), dtype=np.uint8) * 180  # gray background

    # Sky
    scene[:height//2, :] = [220, 200, 170]  # sky color
    # Ground
    scene[height//2:, :] = [150, 170, 130]  # grass color

    # Road
    scene[height//2+30:, 100:540] = [80, 80, 80]
    # Road markings
    for x in range(150, 540, 80):
        scene[height//2+60:height//2+70, x:x+40] = [255, 255, 255]

    objects = []  # list of (x1,y1,x2,y2,class_name)

    # Car 1 (blue)
    cv2.rectangle(scene, (120, 310), (260, 400), (150, 80, 60), -1)   # body
    cv2.rectangle(scene, (150, 280), (235, 315), (130, 70, 50), -1)   # roof
    cv2.circle(scene, (150, 400), 18, (30, 30, 30), -1)               # wheel
    cv2.circle(scene, (230, 400), 18, (30, 30, 30), -1)               # wheel
    cv2.rectangle(scene, (155, 290), (225, 313), (180, 220, 255), -1) # windshield
    objects.append((120, 275, 260, 415, 'car', 0.92))

    # Car 2 (red)
    cv2.rectangle(scene, (350, 315), (480, 400), (50, 60, 180), -1)
    cv2.rectangle(scene, (375, 285), (460, 318), (40, 50, 160), -1)
    cv2.circle(scene, (375, 400), 18, (30, 30, 30), -1)
    cv2.circle(scene, (455, 400), 18, (30, 30, 30), -1)
    cv2.rectangle(scene, (380, 293), (452, 316), (180, 220, 255), -1)
    objects.append((350, 280, 480, 415, 'car', 0.88))

    # Person
    cv2.circle(scene, (540, 290), 20, (200, 170, 140), -1)   # head
    cv2.rectangle(scene, (525, 310), (555, 380), (60, 100, 180), -1)  # body
    cv2.rectangle(scene, (515, 380), (535, 430), (40, 40, 100), -1)   # left leg
    cv2.rectangle(scene, (540, 380), (558, 430), (40, 40, 100), -1)   # right leg
    objects.append((510, 268, 565, 432, 'person', 0.95))

    # Traffic light
    cv2.rectangle(scene, (60, 180), (90, 280), (40, 40, 40), -1)      # pole
    cv2.rectangle(scene, (45, 130), (105, 285), (30, 30, 30), -1)     # housing
    cv2.circle(scene, (75, 155), 18, (0, 0, 220), -1)   # red light
    cv2.circle(scene, (75, 200), 18, (80, 80, 80), -1)  # yellow (off)
    cv2.circle(scene, (75, 245), 18, (80, 80, 80), -1)  # green (off)
    objects.append((45, 128, 105, 290, 'traffic light', 0.89))

    # Dog
    cv2.ellipse(scene, (200, 200), (40, 20), 0, 0, 360, (140, 100, 70), -1)  # body
    cv2.circle(scene,  (235, 188), 18, (150, 110, 80), -1)                    # head
    cv2.ellipse(scene, (248, 182), (8, 5), 30, 0, 360, (120, 90, 60), -1)    # ear
    for lx, ly in [(175, 218), (190, 218), (210, 218), (225, 218)]:
        cv2.rectangle(scene, (lx, ly), (lx+5, ly+20), (130, 95, 65), -1)     # legs
    cv2.ellipse(scene, (163, 200), (12, 6), -30, 0, 360, (140, 100, 70), -1) # tail
    objects.append((158, 178, 258, 240, 'dog', 0.81))

    return scene, objects


scene_img, ground_truth_objects = create_scene()

print(f"Synthetic scene created: {len(ground_truth_objects)} objects")
for obj in ground_truth_objects:
    x1, y1, x2, y2, cls, conf = obj
    print(f"  {cls:15s}: box=({x1},{y1},{x2},{y2}), true confidence={conf}")

In [ ]:
# ── Run YOLO detection ────────────────────────────────────────────────────────

# Save synthetic image to disk for YOLO
cv2.imwrite('/tmp/scene.jpg', scene_img)

if YOLO_AVAILABLE:
    print("Running YOLOv8n inference...")
    t0 = time.time()
    results = model.predict(
        source='/tmp/scene.jpg',
        conf=0.25,          # minimum confidence threshold
        iou=0.45,           # NMS IoU threshold
        device=device,
        verbose=False
    )
    elapsed_ms = (time.time() - t0) * 1000
    print(f"Inference time: {elapsed_ms:.1f}ms")

    result = results[0]   # first (and only) image
    boxes  = result.boxes

    print(f"\nDetected {len(boxes)} objects:")
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        conf     = float(box.conf[0])
        cls_id   = int(box.cls[0])
        cls_name = model.names[cls_id]
        print(f"  [{conf:.3f}] {cls_name:15s} at ({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})")

    # Get annotated image
    annotated = result.plot()
    detected_boxes = [(int(b.xyxy[0][0]), int(b.xyxy[0][1]),
                       int(b.xyxy[0][2]), int(b.xyxy[0][3]),
                       model.names[int(b.cls[0])], float(b.conf[0]))
                      for b in boxes]
else:
    # Simulate detections
    detected_boxes = ground_truth_objects.copy()
    # Slightly perturb boxes to simulate model output
    detected_boxes = [(x1+np.random.randint(-5,5), y1+np.random.randint(-5,5),
                       x2+np.random.randint(-5,5), y2+np.random.randint(-5,5),
                       cls, conf * (0.9 + np.random.rand()*0.15))
                      for x1,y1,x2,y2,cls,conf in detected_boxes]
    print("[SIMULATION] Detected objects:")
    for x1,y1,x2,y2,cls,conf in detected_boxes:
        print(f"  [{conf:.3f}] {cls:15s} at ({x1},{y1},{x2},{y2})")

In [ ]:
# ── Visualize detections ──────────────────────────────────────────────────────

def draw_detections(img, detections, title='YOLO Detections'):
    """Draw bounding boxes and labels on image."""
    vis = img.copy()
    cmap = plt.cm.get_cmap('tab10')
    class_colors = {}

    for x1, y1, x2, y2, cls_name, conf in detections:
        if cls_name not in class_colors:
            idx = len(class_colors) % 10
            r, g, b, _ = cmap(idx)
            class_colors[cls_name] = (int(b*255), int(g*255), int(r*255))  # BGR

        color = class_colors[cls_name]
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)

        label = f'{cls_name} {conf:.2f}'
        (lw, lh), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(vis, (x1, y1-lh-8), (x1+lw+4, y1), color, -1)
        cv2.putText(vis, label, (x1+2, y1-4),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)

    return vis


annotated_manual = draw_detections(scene_img, detected_boxes)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(scene_img, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Scene', fontsize=12); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(annotated_manual, cv2.COLOR_BGR2RGB))
axes[1].set_title('YOLO Detections', fontsize=12); axes[1].axis('off')
plt.tight_layout()
plt.show()

print("\nReading YOLO results:")
print("  result.boxes.xyxy   → [x1, y1, x2, y2] in pixel coordinates")
print("  result.boxes.xywhn  → [x_center, y_center, width, height] normalized [0,1]")
print("  result.boxes.conf   → confidence scores")
print("  result.boxes.cls    → class indices")
print("  result.plot()       → annotated image as numpy array")

---
## 4. Understanding YOLO Output — Boxes, Scores, Classes <a id='4-output'></a>

In [ ]:
# ── Full inference API reference ──────────────────────────────────────────────

print("="*65)
print("COMPLETE ULTRALYTICS INFERENCE API")
print("="*65)

print("""
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

# ─── Predict on various sources ───────────────────────────────────
results = model.predict(
    source='image.jpg',          # image path
    # source='images/',          # folder of images
    # source='video.mp4',        # video file
    # source=0,                  # webcam
    # source='https://...',      # URL
    # source=numpy_array,        # numpy array (H,W,3) BGR

    conf=0.25,                   # minimum confidence (0-1)
    iou=0.45,                    # NMS IoU threshold
    imgsz=640,                   # inference image size
    device='cpu',                # 'cpu', 'cuda', 'mps'
    half=False,                  # FP16 (faster on GPU)
    augment=False,               # test-time augmentation
    save=False,                  # save annotated image to runs/
    save_txt=False,              # save labels to .txt
    verbose=False,               # print per-image summary
    stream=True,                 # generator (for video/large datasets)
)

# ─── Inspect results ──────────────────────────────────────────────
for result in results:
    # Bounding boxes
    boxes = result.boxes
    boxes.xyxy          # tensor (N,4): [x1,y1,x2,y2] absolute pixels
    boxes.xywh          # tensor (N,4): [cx,cy,w,h] absolute pixels
    boxes.xywhn         # tensor (N,4): normalized [0,1]
    boxes.conf          # tensor (N,):  confidence scores
    boxes.cls           # tensor (N,):  class indices

    # Class names
    [result.names[int(c)] for c in boxes.cls]

    # Masks (segmentation model only)
    if result.masks is not None:
        result.masks.data    # tensor (N,H,W) binary masks

    # Keypoints (pose model only)
    if result.keypoints is not None:
        result.keypoints.xy  # tensor (N,K,2)

    # Get annotated image
    annotated = result.plot()   # numpy array (H,W,3) BGR
""")

print("\nConfidence threshold tuning:")
print("  conf=0.10: catch everything (many false positives)")
print("  conf=0.25: default (good balance)")
print("  conf=0.50: conservative (fewer, higher quality detections)")
print("  conf=0.80: very strict (only very confident detections)")

---
## 5. YOLO Tasks — Detection, Segmentation, Classification, Pose <a id='5-tasks'></a>

YOLOv8 supports 5 computer vision tasks with a unified API:

| Task | Model suffix | Output | Example |
|---|---|---|---|
| Detection | `yolov8n.pt` | Bounding boxes | Car at (x,y,w,h) |
| Segmentation | `yolov8n-seg.pt` | Boxes + pixel masks | Exact shape of car |
| Classification | `yolov8n-cls.pt` | Class probabilities | 92% cat, 5% dog |
| Pose estimation | `yolov8n-pose.pt` | Keypoints (joints) | 17 body keypoints |
| OBB (Oriented) | `yolov8n-obb.pt` | Rotated boxes | Satellite aerial objects |

In [ ]:
# ── YOLO tasks overview with code ─────────────────────────────────────────────

print("="*65)
print("YOLO TASK EXAMPLES")
print("="*65)

tasks = [
    ("1. Object Detection",
     "yolov8n.pt",
     """model = YOLO('yolov8n.pt')
results = model('image.jpg')
for box in results[0].boxes:
    x1,y1,x2,y2 = box.xyxy[0]
    cls = model.names[int(box.cls[0])]
    conf = float(box.conf[0])
    print(f'{cls}: {conf:.2f} at ({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})')"""),

    ("2. Instance Segmentation",
     "yolov8n-seg.pt",
     """model = YOLO('yolov8n-seg.pt')
results = model('image.jpg')
masks = results[0].masks.data    # (N, H, W) binary masks
# Each mask covers the exact pixels of one detected object"""),

    ("3. Image Classification",
     "yolov8n-cls.pt",
     """model = YOLO('yolov8n-cls.pt')
results = model('image.jpg')
probs = results[0].probs
top1  = model.names[probs.top1]    # top-1 class name
top5  = [model.names[i] for i in probs.top5]  # top-5"""),

    ("4. Pose Estimation",
     "yolov8n-pose.pt",
     """model = YOLO('yolov8n-pose.pt')
results = model('image.jpg')
kpts = results[0].keypoints.xy   # (N_persons, 17_joints, 2)
# COCO 17 keypoints: nose,eyes,ears,shoulders,elbows,wrists,..."""),
]

for name, weights, code in tasks:
    print(f"\n{'─'*60}")
    print(f"  {name}  (weights: {weights})")
    print(f"{'─'*60}")
    for line in code.split('\n'):
        print(f"  {line}")

print("\n" + "═"*65)
print("All tasks use identical API — just swap the model file!")
print("  Detection: .pt")
print("  Segmentation: -seg.pt")
print("  Classification: -cls.pt")
print("  Pose: -pose.pt")

In [ ]:
# ── Visualize all tasks on synthetic data ────────────────────────────────────

# Create synthetic images showing what each task outputs
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
titles = ['Detection\n(boxes)', 'Segmentation\n(boxes + masks)',
          'Classification\n(class probabilities)', 'Pose Estimation\n(keypoints)']

# Detection
detect_img = scene_img.copy()
detected_vis = draw_detections(detect_img, detected_boxes[:3])
axes[0].imshow(cv2.cvtColor(detected_vis, cv2.COLOR_BGR2RGB))
axes[0].set_title(titles[0], fontsize=10); axes[0].axis('off')

# Segmentation (simulated mask)
seg_img = scene_img.copy()
overlay = np.zeros_like(seg_img)
# Simulate pixel masks for each detection
mask_colors = [(255,0,0,120), (0,255,0,120), (0,0,255,120)]
for (x1,y1,x2,y2,cls,conf), (r,g,b,a) in zip(detected_boxes[:3], mask_colors):
    cv2.rectangle(overlay, (x1,y1), (x2,y2), (b,g,r), -1)
seg_vis = cv2.addWeighted(seg_img, 1.0, overlay, 0.4, 0)
detected_vis2 = draw_detections(seg_vis, detected_boxes[:3])
axes[1].imshow(cv2.cvtColor(detected_vis2, cv2.COLOR_BGR2RGB))
axes[1].set_title(titles[1], fontsize=10); axes[1].axis('off')

# Classification
cls_img = np.ones((480, 640, 3), dtype=np.uint8) * 240
cls_probs = [('car', 0.83), ('truck', 0.07), ('bus', 0.05), ('bicycle', 0.03), ('motorcycle', 0.02)]
for i, (cls_name, prob) in enumerate(cls_probs):
    y = 80 + i * 70
    bar_w = int(prob * 400)
    cv2.rectangle(cls_img, (120, y-15), (120+bar_w, y+15), (50, 150, 50), -1)
    cv2.putText(cls_img, f'{cls_name}: {prob:.0%}', (10, y+7),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (30,30,30), 2)
axes[2].imshow(cv2.cvtColor(cls_img, cv2.COLOR_BGR2RGB))
axes[2].set_title(titles[2], fontsize=10); axes[2].axis('off')

# Pose estimation
pose_img = np.ones((480, 640, 3), dtype=np.uint8) * 200
# Draw stick figure
keypoints = {
    'nose': (320, 80), 'l_eye': (305, 70), 'r_eye': (335, 70),
    'l_ear': (295, 75), 'r_ear': (345, 75),
    'l_shoulder': (280, 130), 'r_shoulder': (360, 130),
    'l_elbow': (250, 190), 'r_elbow': (390, 190),
    'l_wrist': (230, 250), 'r_wrist': (410, 250),
    'l_hip': (295, 250), 'r_hip': (345, 250),
    'l_knee': (285, 340), 'r_knee': (355, 340),
    'l_ankle': (280, 420), 'r_ankle': (360, 420),
}
skeleton = [('nose','l_shoulder'),('nose','r_shoulder'),
            ('l_shoulder','r_shoulder'),('l_shoulder','l_elbow'),
            ('l_elbow','l_wrist'),('r_shoulder','r_elbow'),
            ('r_elbow','r_wrist'),('l_shoulder','l_hip'),
            ('r_shoulder','r_hip'),('l_hip','r_hip'),
            ('l_hip','l_knee'),('l_knee','l_ankle'),
            ('r_hip','r_knee'),('r_knee','r_ankle')]
for a, b in skeleton:
    cv2.line(pose_img, keypoints[a], keypoints[b], (0, 120, 255), 2)
for name, pt in keypoints.items():
    cv2.circle(pose_img, pt, 6, (255, 50, 50), -1)
axes[3].imshow(cv2.cvtColor(pose_img, cv2.COLOR_BGR2RGB))
axes[3].set_title(titles[3], fontsize=10); axes[3].axis('off')

plt.suptitle('YOLO Task Overview — Same Model Family, Different Outputs', fontsize=13)
plt.tight_layout()
plt.show()

---
## 6. Training YOLO on Custom Data <a id='6-training'></a>

In [ ]:
# ── YOLO custom training guide ────────────────────────────────────────────────

print("="*65)
print("TRAINING YOLO ON CUSTOM DATA — STEP BY STEP")
print("="*65)

print("""
STEP 1: Prepare your dataset
─────────────────────────────
data/
├── images/
│   ├── train/   image1.jpg, image2.jpg, ...
│   └── val/     image500.jpg, ...
└── labels/
    ├── train/   image1.txt, image2.txt, ...
    └── val/     image500.txt, ...

Each .txt label file: one row per object
Format: <class_id> <x_center> <y_center> <width> <height>
All values normalized [0, 1] relative to image dimensions

Example label file (image1.txt):
  0 0.45 0.62 0.30 0.40   ← class 0 (car), center x=45%, y=62%, w=30%, h=40%
  1 0.80 0.30 0.10 0.25   ← class 1 (person)


STEP 2: Create dataset YAML config
───────────────────────────────────
# my_dataset.yaml
path: /path/to/data          # root dir
train: images/train          # relative to path
val:   images/val
nc: 3                        # number of classes
names: ['car', 'person', 'dog']   # class names


STEP 3: Train
──────────────
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # Start from pre-trained weights

results = model.train(
    data='my_dataset.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device='cuda',
    workers=8,
    patience=20,             # early stopping if no improvement
    save_period=10,          # save checkpoint every 10 epochs
    project='runs/train',
    name='my_model',
    optimizer='Adam',
    lr0=0.01,                # initial LR
    augment=True,            # mosaic, flips, etc.
)


STEP 4: Evaluate
─────────────────
model.val(data='my_dataset.yaml')
# Prints: mAP50, mAP50-95, precision, recall per class


STEP 5: Inference with custom model
─────────────────────────────────────
model = YOLO('runs/train/my_model/weights/best.pt')
results = model.predict('new_image.jpg')
""")

print("\nData annotation tools (free):")
print("  • Label Studio:  https://labelstud.io/")
print("  • CVAT:          https://www.cvat.ai/")
print("  • Roboflow:      https://roboflow.com/ (also free tier)")
print("  • LabelImg:      pip install labelImg")
print()
print("Minimum dataset size recommendations:")
print("  • Simple objects, controlled background: 100–300 images per class")
print("  • Complex real-world scenes: 1,000–5,000 images per class")
print("  • Use Roboflow Universe to find public datasets: https://universe.roboflow.com/")

In [ ]:
# ── Training metrics explained ────────────────────────────────────────────────

# Simulate training curve data
epochs_sim = np.arange(1, 51)

# Simulated realistic training curves
np.random.seed(42)
box_loss  = 1.5 * np.exp(-epochs_sim/15) + 0.3 + np.random.randn(50)*0.02
cls_loss  = 1.2 * np.exp(-epochs_sim/12) + 0.2 + np.random.randn(50)*0.02
map50     = 0.85 * (1 - np.exp(-epochs_sim/10)) + np.random.randn(50)*0.01
map50_95  = 0.55 * (1 - np.exp(-epochs_sim/12)) + np.random.randn(50)*0.01
precision = 0.82 * (1 - np.exp(-epochs_sim/8))  + 0.08 + np.random.randn(50)*0.01
recall    = 0.78 * (1 - np.exp(-epochs_sim/11)) + 0.10 + np.random.randn(50)*0.01

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

metrics = [
    (axes[0,0], box_loss,  'Box Loss',  'Loss', '#e74c3c'),
    (axes[0,1], cls_loss,  'Class Loss','Loss', '#e67e22'),
    (axes[0,2], map50,     'mAP@0.5',  'mAP',  '#27ae60'),
    (axes[1,0], map50_95,  'mAP@0.5:0.95','mAP','#2ecc71'),
    (axes[1,1], precision, 'Precision', 'Score','#3498db'),
    (axes[1,2], recall,    'Recall',   'Score', '#9b59b6'),
]

for ax, data, title, ylabel, color in metrics:
    ax.plot(epochs_sim, data, color=color, lw=2)
    ax.fill_between(epochs_sim, data, alpha=0.15, color=color)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)

plt.suptitle('Typical YOLO Training Curves — 50 Epochs on Custom Dataset', fontsize=13)
plt.tight_layout()
plt.show()

print("Metric explanations:")
print("  Box Loss:   how accurate are the bounding box coordinates?")
print("  Class Loss: how accurate are the class predictions?")
print("  mAP@0.5:    mean Average Precision at IoU threshold 0.50")
print("              (the standard 'detection accuracy' metric)")
print("  mAP@0.5:0.95: averaged over IoU thresholds 0.50–0.95 (stricter)")
print("  Precision:  of all detected boxes, what % are correct?")
print("  Recall:     of all real objects, what % did we detect?")

---
## 7. Model Export & Deployment <a id='7-export'></a>

In [ ]:
print("="*65)
print("YOLO EXPORT & DEPLOYMENT")
print("="*65)

print("""
# Export to different formats
model = YOLO('yolov8n.pt')

model.export(format='onnx')      # ONNX — cross-platform, most compatible
model.export(format='torchscript') # TorchScript — PyTorch production
model.export(format='tflite')    # TensorFlow Lite — Android/iOS
model.export(format='coreml')    # CoreML — Apple devices
model.export(format='engine')    # TensorRT — NVIDIA GPU (fastest)
model.export(format='openvino')  # OpenVINO — Intel CPU/VPU


# Use exported model for inference
onnx_model = YOLO('yolov8n.onnx')
results = onnx_model.predict('image.jpg')


# Benchmarking exported models
from ultralytics.utils.benchmarks import benchmark
benchmark(
    model='yolov8n.pt',
    data='coco8.yaml',
    imgsz=640,
    half=False
)
""")

print("Format comparison:")
formats = [
    ('PyTorch (.pt)',        'Training, research',   '1×',  'Full PyTorch required'),
    ('ONNX (.onnx)',         'Cross-platform',       '1.5×','ONNX Runtime (~20MB)'),
    ('TensorRT (.engine)',   'NVIDIA GPU production','5-8×','NVIDIA GPU only'),
    ('TFLite (.tflite)',     'Mobile/embedded',      '2-3×','Very small runtime'),
    ('CoreML (.mlmodel)',    'Apple devices',         '3×', 'macOS/iOS only'),
    ('OpenVINO',             'Intel CPU/VPU',        '2-4×','Intel hardware'),
]
print(f"\n{'Format':22s} {'Use Case':22s} {'Speed':8s} {'Requirement'}")
print("─" * 70)
for fmt, use, speed, req in formats:
    print(f"{fmt:22s} {use:22s} {speed:8s} {req}")

---
## 8. Mini Project — Vehicle Counter for Traffic Analysis <a id='8-mini-project'></a>

### What We're Building

A system that:
1. Detects all vehicles in each frame of a video
2. Tracks them across frames using a virtual counting line
3. Counts how many vehicles crossed the line
4. Produces traffic analytics (vehicles per minute, type breakdown)

**Real-world use:** Smart city traffic management, toll booth automation, parking lot occupancy.

In [ ]:
# ── Simulate vehicle tracking across video frames ─────────────────────────────

np.random.seed(7)

VEHICLE_CLASSES = {'car': 2, 'truck': 7, 'bus': 5, 'motorcycle': 3}
COLORS = {'car': (50, 150, 255), 'truck': (255, 100, 50),
          'bus': (50, 200, 50), 'motorcycle': (200, 50, 200)}


class VehicleCounter:
    """
    Count vehicles crossing a virtual line in video frames.
    
    Algorithm:
        1. Detect vehicles in each frame (YOLO)
        2. Track each detection: associate with previous frame's boxes
        3. When a box center crosses the counting line → count +1
    """

    def __init__(self, count_line_y, img_width):
        self.count_line_y = count_line_y
        self.img_width    = img_width
        self.counts       = {cls: 0 for cls in VEHICLE_CLASSES}
        self.prev_centers = {}   # track_id → (prev_y, class)
        self.next_id      = 0
        self.frame_data   = []   # for analysis

    def process_frame(self, detections, frame_num):
        """
        Process one frame's detections.
        detections: list of (x1,y1,x2,y2,class_name,confidence)
        """
        current_centers = []
        crossed_this_frame = []

        for x1, y1, x2, y2, cls_name, conf in detections:
            cx = (x1 + x2) // 2
            cy = (y1 + y2) // 2
            current_centers.append((cx, cy, cls_name))

        # Simple tracking: match by nearest previous center
        matched = set()
        new_prev = {}

        for cx, cy, cls_name in current_centers:
            best_id = None
            best_dist = 80  # max pixel distance to match

            for tid, (prev_y, prev_cls) in self.prev_centers.items():
                if tid in matched or prev_cls != cls_name:
                    continue
                dist = abs(cy - prev_y)
                if dist < best_dist:
                    best_dist = dist
                    best_id = tid

            if best_id is None:
                best_id = self.next_id
                self.next_id += 1

            matched.add(best_id)

            # Check if crossed counting line
            if best_id in self.prev_centers:
                prev_y, _ = self.prev_centers[best_id]
                if prev_y < self.count_line_y <= cy:   # crossed downward
                    if cls_name in self.counts:
                        self.counts[cls_name] += 1
                    crossed_this_frame.append(cls_name)

            new_prev[best_id] = (cy, cls_name)

        self.prev_centers = new_prev
        total = sum(self.counts.values())
        self.frame_data.append({'frame': frame_num, 'total': total, 'counts': dict(self.counts)})

        return crossed_this_frame

    def report(self):
        total = sum(self.counts.values())
        print("\n" + "═"*50)
        print("  TRAFFIC ANALYSIS REPORT")
        print("═"*50)
        print(f"  Total vehicles counted: {total}")
        for cls_name, count in self.counts.items():
            pct = count/max(1,total)*100
            bar = '█' * int(pct / 3)
            print(f"  {cls_name:12s}: {count:3d} ({pct:.1f}%) {bar}")


# ── Simulate 60 video frames ──────────────────────────────────────────────────

FRAME_W, FRAME_H = 640, 480
COUNT_LINE_Y = 300

counter = VehicleCounter(count_line_y=COUNT_LINE_Y, img_width=FRAME_W)

# Simulate vehicles moving down the frame
vehicle_tracks = []
for _ in range(15):  # 15 vehicles
    cls = np.random.choice(list(VEHICLE_CLASSES.keys()), p=[0.6, 0.2, 0.1, 0.1])
    start_frame = np.random.randint(0, 40)
    x = np.random.randint(50, 550)
    w = {'car': 80, 'truck': 100, 'bus': 110, 'motorcycle': 50}[cls]
    h = {'car': 50, 'truck': 70,  'bus': 80,  'motorcycle': 40}[cls]
    speed = np.random.randint(5, 12)  # pixels per frame
    vehicle_tracks.append((cls, start_frame, x, w, h, speed))

all_frames_detections = []
for frame_num in range(60):
    detections = []
    for cls, start, x, w, h, speed in vehicle_tracks:
        y = 50 + (frame_num - start) * speed
        if start <= frame_num and 0 <= y <= FRAME_H + h:
            conf = 0.75 + np.random.rand() * 0.2
            detections.append((x, y, x+w, y+h, cls, conf))
    all_frames_detections.append(detections)
    counter.process_frame(detections, frame_num)

counter.report()

# Visualize selected frames
selected = [0, 15, 30, 45]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, frame_num in zip(axes, selected):
    frame = np.ones((FRAME_H, FRAME_W, 3), dtype=np.uint8) * 160
    # Road
    frame[200:, 50:590] = [80, 80, 80]
    # Lane markings
    for x in range(100, 600, 120):
        frame[230:240, x:x+60] = [255, 255, 255]

    # Draw counting line
    cv2.line(frame, (0, COUNT_LINE_Y), (FRAME_W, COUNT_LINE_Y), (0, 255, 255), 3)
    cv2.putText(frame, 'COUNT LINE', (10, COUNT_LINE_Y-8),
               cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,255), 1)

    # Draw vehicles
    for x1, y1, x2, y2, cls_name, conf in all_frames_detections[frame_num]:
        color = COLORS[cls_name]
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        label = f'{cls_name} {conf:.2f}'
        cv2.putText(frame, label, (x1, y1-5),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

    # Count overlay
    total_counted = counter.frame_data[frame_num]['total']
    cv2.rectangle(frame, (0, 0), (200, 30), (0,0,0), -1)
    cv2.putText(frame, f'Frame {frame_num} | Counted: {total_counted}',
               (5, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1)

    ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Frame {frame_num} — {len(all_frames_detections[frame_num])} vehicles', fontsize=9)
    ax.axis('off')

plt.suptitle('Vehicle Counting System — Simulated Traffic Camera\n'
             'Cyan line = counting trigger; vehicles counted when they cross it', fontsize=12)
plt.tight_layout()
plt.show()

# Plot cumulative count
fig, ax = plt.subplots(figsize=(10, 4))
frames = [d['frame'] for d in counter.frame_data]
totals = [d['total'] for d in counter.frame_data]
ax.step(frames, totals, 'b-', lw=2, label='Cumulative count')
ax.fill_between(frames, totals, step='pre', alpha=0.15)
ax.set_xlabel('Frame number'); ax.set_ylabel('Vehicles counted')
ax.set_title('Cumulative Vehicle Count Over Time')
ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Common Pitfalls <a id='9-pitfalls'></a>

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║              YOLO ULTRALYTICS — COMMON PITFALLS                  ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. WRONG: Setting conf=0.9 (too high) → missing detections     ║
║     results = model.predict(img, conf=0.9)  ← misses 40% objects ║
║  RIGHT: Start with conf=0.25 (default), tune on your data        ║
║                                                                  ║
║  2. WRONG: Not using stream=True for video/large batches         ║
║     results = model.predict('video.mp4')   ← loads ALL frames!  ║
║  RIGHT: Use generator to process frame-by-frame                  ║
║     for result in model.predict('video.mp4', stream=True):       ║
║         process(result)    ← memory efficient                    ║
║                                                                  ║
║  3. WRONG: Using detection model for classification              ║
║     model = YOLO('yolov8n.pt')  ← detection model               ║
║     # Output: boxes (x1,y1,x2,y2,conf,cls) for each object      ║
║     # NOT image-level class probability!                         ║
║  RIGHT for classification: YOLO('yolov8n-cls.pt')               ║
║                                                                  ║
║  4. WRONG: YOLO label format error (absolute vs normalized)      ║
║     label: 0 120 80 200 150   ← absolute pixel coords → WRONG    ║
║  RIGHT: All values MUST be normalized [0,1]                      ║
║     label: 0 0.50 0.33 0.31 0.31   ← normalized                 ║
║                                                                  ║
║  5. WRONG: Not balancing dataset classes                         ║
║     1000 images of cars, 50 of pedestrians                       ║
║     → model learns to ignore pedestrians!                        ║
║  RIGHT: Balance classes or use class_weights in training         ║
║                                                                  ║
║  6. WRONG: Ignoring NMS threshold                               ║
║     Default iou=0.7 may produce duplicate detections             ║
║     on crowded scenes (touching objects)                         ║
║  RIGHT: Lower iou=0.4–0.5 to remove more duplicates             ║
║                                                                  ║
║  7. WRONG: Evaluating with mAP@0.5 only for safety-critical apps ║
║     mAP50 misses poorly localized boxes (IoU=0.55 looks good)   ║
║  RIGHT: Use mAP50-95 and check per-class recalls separately      ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")

---
## 10. Interview Q&A <a id='10-interview'></a>

In [ ]:
qa = [
    ("What does 'You Only Look Once' mean and how is it different from older detectors?",
     "YOLO processes the entire image in ONE forward pass through a single neural network, "
     "outputting all bounding boxes and class scores simultaneously. Older methods: "
     "(1) Sliding window: scan image with a fixed window at multiple positions/scales — "
     "very slow O(scales × positions). (2) R-CNN: extract ~2000 region proposals first, "
     "then classify each separately — still slow. (3) Faster R-CNN: predict proposals with "
     "a Region Proposal Network, still 2-stage. YOLO: 1-stage, single pass, "
     "30-100 FPS vs 5-15 FPS for Faster R-CNN."),

    ("What is mAP (mean Average Precision) and how is it calculated?",
     "mAP measures detection model quality by averaging precision across different recall levels. "
     "For each class: (1) rank predictions by confidence; (2) compute precision-recall curve "
     "(TP+FP=all detections, TP+FN=all ground truths); (3) Average Precision = area under PR curve. "
     "mAP = average AP across all classes. "
     "mAP@0.5: detections with IoU≥0.5 are correct. "
     "mAP@0.5:0.95: average over IoU thresholds 0.50, 0.55, ..., 0.95 (COCO standard, harder)."),

    ("What is NMS (Non-Maximum Suppression) and why is it needed?",
     "YOLO predicts one box per grid cell, and multiple cells often detect the same object. "
     "NMS removes duplicate predictions: (1) Sort all boxes by confidence score. "
     "(2) Keep the highest-confidence box. (3) Remove all boxes with IoU > threshold "
     "(typically 0.45) with the kept box. (4) Repeat. Without NMS, you'd get 3-5 boxes "
     "for each object. The IoU threshold controls aggressiveness: lower = more removal "
     "(fewer boxes), higher = more permissive (more duplicates allowed)."),

    ("When would you use YOLO vs Faster R-CNN?",
     "YOLO: real-time applications (video surveillance, autonomous driving, robotics), "
     "mobile/edge deployment, when speed > absolute accuracy, batch processing. "
     "Faster R-CNN: when accuracy is paramount (medical imaging, satellite analysis), "
     "small dense objects (crowded scenes), when you can afford slower inference. "
     "Modern YOLO (v8/v11) has largely closed the accuracy gap while maintaining "
     "3-10× speed advantage. Faster R-CNN is mainly used in research and "
     "very high-precision applications."),

    ("How do you prepare data for YOLO custom training?",
     "YOLO expects: (1) Images in any standard format (JPEG, PNG). "
     "(2) Label .txt files with same name as image, one line per object: "
     "'class_id x_center y_center width height' all normalized [0,1]. "
     "(3) A YAML config with paths and class names. "
     "Tools: Label Studio, CVAT, Roboflow (auto-converts from any format). "
     "Minimum ~100 images per class for simple objects; 1000+ for complex real-world scenes. "
     "Split: typically 80% train / 10% val / 10% test."),

    ("What is the difference between detection and segmentation in YOLO?",
     "Detection (yolov8n.pt): outputs bounding BOXES — axis-aligned rectangles "
     "around each detected object. Fast and sufficient for most tasks. "
     "Segmentation (yolov8n-seg.pt): outputs bounding boxes PLUS pixel-level MASKS "
     "showing the exact shape of each object (not just a rectangle). "
     "Use segmentation when: you need precise shape (counting pixels, measuring area), "
     "object shapes matter (oddly-shaped objects), background removal, "
     "or AR/VFX overlays. ~20% slower than detection."),
]

print("=" * 70)
print("  INTERVIEW Q&A — YOLO / OBJECT DETECTION")
print("=" * 70)
for i, (q, a) in enumerate(qa, 1):
    print(f"\nQ{i}: {q}")
    print(f"\nA{i}: {a}")
    print("\n" + "─" * 70)

---
## 11. Resources <a id='11-resources'></a>

### Official Documentation
- **Ultralytics YOLO**: https://docs.ultralytics.com/
- **YOLOv8 predict**: https://docs.ultralytics.com/modes/predict/
- **YOLOv8 train**: https://docs.ultralytics.com/modes/train/

### Research Papers
- **YOLO original** (Redmon et al., 2016): https://arxiv.org/abs/1506.02640
- **YOLOv4**: https://arxiv.org/abs/2004.10934
- **YOLOv8 blog post**: https://docs.ultralytics.com/models/yolov8/

### Video Tutorials
- **YOLOv8 full tutorial** (Nicholas Renotte): https://youtu.be/wuZtUMEiKWY
- **Train YOLO on custom data**: https://youtu.be/m9fH9OWn8YM
- **Object tracking with YOLO**: https://youtu.be/OS5qI9YBkfk

### Datasets
- **Roboflow Universe** (free datasets): https://universe.roboflow.com/
- **COCO Dataset**: https://cocodataset.org/
- **Open Images**: https://storage.googleapis.com/openimages/web/index.html

### Tools
- **Label Studio** (annotation): https://labelstud.io/
- **Roboflow** (annotation + augmentation): https://roboflow.com/
- **Supervision** (post-processing utilities): https://supervision.roboflow.com/

---
## 12. Summary & What's Next <a id='12-summary'></a>

### What You Learned

| Concept | Key Takeaway |
|---|---|
| YOLO architecture | One forward pass → all boxes; backbone + neck + head + NMS |
| IoU | Overlap metric; IoU≥0.5 = correct detection |
| NMS | Remove duplicate boxes; lower threshold = stricter |
| Inference API | `model.predict(source, conf, iou, device)` — 3 lines |
| Output format | `result.boxes.xyxy`, `.conf`, `.cls`, `result.plot()` |
| 5 YOLO tasks | Detection, Segmentation, Classification, Pose, OBB |
| Custom training | YAML config + normalized labels + `model.train()` |
| mAP | Primary metric — higher is better; mAP50-95 is stricter |
| Export | ONNX, TensorRT, TFLite for different deployment targets |

### What's Next

| Notebook | Topic |
|---|---|
| `Detectron2` | Facebook AI research framework — Mask R-CNN, instance segmentation, keypoints |

**YOLO answers "where + what?" for every object in real time. Detectron2 adds pixel-perfect masks!**